# Stable Diffusion + Tiled Noise-Attack Gate

This notebook joins the two things I built earlier in this repo, plus the reference paper:

1. **The Stable Diffusion workflow** — from the Hugging Face `diffusers` walkthrough
   ([`../Resources/stable_diffusion.ipynb`](../Resources/stable_diffusion.ipynb)):
   `StableDiffusionPipeline` for generation, and the exposed components
   (VAE / UNet / scheduler) for the training path `image -> VAE.encode -> add noise -> UNet`.
2. **The tiled noise-attack detection I practiced** in
   [`../04-Adversarial-Attacks/tiled_noise_attack_detection.ipynb`](../04-Adversarial-Attacks/tiled_noise_attack_detection.ipynb):
   attack an irregular region of an image with **FGSM/PGD**, cut the image into a grid of
   tiles with `tile_image`, score each tile's high-frequency energy, and flag the tiles
   that carry injected noise (threshold `median + k*MAD` calibrated on clean tiles).
3. **The paper** — **ViT-ReciproCAM** ([arXiv:2310.02588](https://arxiv.org/abs/2310.02588)),
   gradient-free visual explanations via spatial masking. Its key lesson used here:
   **forward-only scoring needs no backprop, so a whole stack of tiles is checked in one
   GPU batch** — cheap enough to run on *every* image entering training.

**What the gate does.** Stable Diffusion is trained on scraped images. If an attacker
poisons some of them with imperceptible adversarial noise, the model learns from corrupted
data. This notebook puts the tile detector **in front of the SD training path as a module**:
every incoming image is tiled and scanned; poisoned images are **quarantined before
`VAE.encode`**, and the flagged tiles show *where* the noise is.

```
incoming image --> [ TILE + SCAN (the module) ] --keep--> VAE.encode -> add noise -> UNet
                              |
                              +--reject--> quarantine (never reaches training)
```

> **Kaggle:** set accelerator to **GPU T4 x2** and turn **Internet On**.
> Generation runs on one T4; the tile scan is batched across **both** T4s.


## 0. Install & imports


In [ ]:
!pip install -q diffusers transformers accelerate


In [ ]:
# ============================================================
# CELL 1 - Imports & GPU inventory
# ============================================================
import torch, torch.nn.functional as F
import numpy as np, matplotlib.pyplot as plt
import matplotlib.patches as patches
from torchvision import models, transforms
from PIL import Image, ImageDraw
import urllib.request, os
from concurrent.futures import ThreadPoolExecutor

N_GPU = torch.cuda.device_count()
DEVICES = [f"cuda:{i}" for i in range(N_GPU)] if N_GPU else ["cpu"]
print("GPUs found:", N_GPU)
for i in range(N_GPU):
    print(f"  cuda:{i} -> {torch.cuda.get_device_name(i)}")
print("Using devices:", DEVICES)


## 1. The detection module (from my practiced notebook)

Copied from [`tiled_noise_attack_detection.ipynb`](../04-Adversarial-Attacks/tiled_noise_attack_detection.ipynb),
only resized to SD's native **512x512** (so `GRID=4` gives 128-px tiles).

- Images are `[1,3,SIZE,SIZE]` float tensors in **[0,1] pixel space** — noise budgets stay in real pixels.
- `tile_image` cuts one image into a row-major `[16,3,128,128]` stack.
- `hf_energy` = `mean(|tile - blur(tile)|)` — FGSM/PGD noise is high-frequency.
- `clean_threshold` = `median + k*1.4826*MAD` over **trusted clean** tiles, calibrated **once**.
- The scan is **model-free and forward-only** (the ViT-ReciproCAM lesson), so all tiles of
  all images go through as one batch, split across both T4s.


In [ ]:
# ============================================================
# CELL 2 - Tiler + per-tile detector (reused from 04-Adversarial-Attacks)
# ============================================================
SIZE = 512            # Stable Diffusion native resolution
GRID = 4              # 4x4 = 16 tiles per image
TILE = SIZE // GRID   # 128 px per tile

to_tensor = transforms.Compose([transforms.Resize((SIZE, SIZE)), transforms.ToTensor()])

def load_image(path_or_url):
    # local path OR url -> [1,3,SIZE,SIZE] float tensor in [0,1] on CPU
    if str(path_or_url).startswith("http"):
        fn = "/tmp/" + os.path.basename(path_or_url)
        if not os.path.exists(fn):
            urllib.request.urlretrieve(path_or_url, fn)
        path_or_url = fn
    return to_tensor(Image.open(path_or_url).convert("RGB")).unsqueeze(0)

def pil_to_tensor(img):
    # PIL image (e.g. straight out of the SD pipeline) -> [1,3,SIZE,SIZE] in [0,1]
    return to_tensor(img.convert("RGB")).unsqueeze(0)

# ---- THE TILE FUNCTION (unchanged from the practiced notebook) ----
def tile_image(x, grid=GRID, tile=TILE):
    # [1,3,SIZE,SIZE] -> [grid*grid, 3, tile, tile]  (row-major tile order)
    p = x.unfold(2, tile, tile).unfold(3, tile, tile)
    p = p.permute(0, 2, 3, 1, 4, 5).reshape(-1, 3, tile, tile)
    return p.contiguous()

def tile_grid_to_full(per_tile, grid=GRID, tile=TILE):
    # [grid*grid] per-tile values -> [SIZE,SIZE] blocky heatmap for overlay
    g = np.asarray(per_tile).reshape(grid, grid)
    return np.kron(g, np.ones((tile, tile)))

def to_np(t):   return t.squeeze().detach().cpu().permute(1, 2, 0).numpy()
def norm01(a):
    a = np.asarray(a, np.float32); return (a - a.min()) / (a.max() - a.min() + 1e-8)

# ---- The high-frequency-energy score ----
def _gaussian_kernel(sigma=1.0, ksize=5):
    ax = torch.arange(ksize) - ksize // 2
    g = torch.exp(-(ax**2) / (2*sigma**2)); g = g / g.sum()
    return torch.outer(g, g).view(1, 1, ksize, ksize).repeat(3, 1, 1, 1)  # depthwise, 3ch
_GK = _gaussian_kernel()

def hf_energy(tiles):
    # tiles:[B,3,h,w] in [0,1] on ANY device -> [B] numpy score (one per tile)
    k = _GK.to(tiles.device, tiles.dtype)
    blur = F.conv2d(tiles, k, padding=k.shape[-1]//2, groups=3)
    return (tiles - blur).abs().mean(dim=(1, 2, 3)).float().cpu().numpy()

def batched_scan(images, use_gpus=None):
    # Tile EVERY image, stack all tiles into one [N*16,3,128,128] batch,
    # split it across the available GPUs, score in parallel (forward-only, no grads).
    devs = use_gpus or DEVICES
    all_tiles = torch.cat([tile_image(x) for x in images])          # [N*16,3,TILE,TILE]
    chunks = torch.chunk(all_tiles, len(devs), dim=0) if len(devs) > 1 else [all_tiles]
    def _run(args):
        d, ch = args
        with torch.no_grad():        # no_grad is thread-local -> re-enter in each worker
            return hf_energy(ch.to(d))
    with ThreadPoolExecutor(max_workers=len(devs)) as ex:
        outs = list(ex.map(_run, zip(devs[:len(chunks)], chunks)))
    scores = np.concatenate(outs)
    return scores.reshape(len(images), GRID*GRID)                    # [N,16]

def clean_threshold(clean_images, k=3.0):
    # Calibrate ONCE on trusted clean images: robust median + k*MAD baseline
    s = batched_scan(clean_images).ravel()
    med = np.median(s); mad = np.median(np.abs(s - med)) + 1e-8
    return med + k * 1.4826 * mad

print(f"module ready: {GRID}x{GRID} grid, {TILE}x{TILE}px tiles, {GRID*GRID} tiles/image")


## 2. The attack I practiced (FGSM / PGD on an irregular region)

Unchanged from the practiced notebook: a ResNet50 supplies the gradient, the perturbation
is confined to a **random irregular blob** (not tile-aligned — finding *which tiles* it
touched is the tiler's job), and the image stays a valid [0,1] picture inside an
L-infinity `epsilon` budget. **The gate itself never uses this model** — it is only how
we manufacture poisoned test images.


In [ ]:
# ============================================================
# CELL 3 - Irregular blob mask + FGSM/PGD poison (reused from 04-Adversarial-Attacks)
# ============================================================
_mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
_std  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
def normalize(t): return (t - _mean.to(t.device)) / _std.to(t.device)

_D = DEVICES[0]
_ATK = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2).eval().to(_D)

def random_blob_mask(size=SIZE, seed=None, n_verts=9, rad_frac=(0.12, 0.34)):
    # Irregular, non-rectangular polygon -> [1,1,size,size] in {0,1}
    rng = np.random.default_rng(seed)
    cx, cy = rng.uniform(0.30, 0.70, 2) * size
    angles = np.sort(rng.uniform(0, 2*np.pi, n_verts))
    radii = rng.uniform(*rad_frac, n_verts) * size
    pts = [(float(cx + r*np.cos(a)), float(cy + r*np.sin(a))) for a, r in zip(angles, radii)]
    im = Image.new("L", (size, size), 0); ImageDraw.Draw(im).polygon(pts, fill=1)
    return torch.tensor(np.array(im), dtype=torch.float32).view(1, 1, size, size)

def _grad_sign(xi):
    out = _ATK(normalize(xi))
    loss = F.cross_entropy(out, out.argmax(1).detach())  # push away from current prediction
    _ATK.zero_grad(); loss.backward()
    return xi.grad.sign()

def poison(x, method="pgd", epsilon=0.05, alpha=0.01, steps=10, mask=None):
    # FGSM (1 step) or PGD (iterative, subtler) confined to the blob mask. Returns CPU tensor.
    x0 = x.clone().detach().to(_D); xi = x0.clone()
    m = mask.to(_D) if mask is not None else None
    n_steps, step_size = (1, epsilon) if method == "fgsm" else (steps, alpha)
    for _ in range(n_steps):
        xi.requires_grad_(True)
        step = step_size * _grad_sign(xi)
        if m is not None: step = step * m
        with torch.no_grad():
            xi = torch.min(torch.max(xi.detach() + step, x0 - epsilon), x0 + epsilon)
            xi = torch.clamp(xi, 0, 1)
    return xi.detach().cpu()

def true_tiles_from_mask(mask, cover=0.05):
    # ground truth: a tile is 'truly attacked' if the blob covers > cover of it
    mt = tile_image(mask.repeat(1, 3, 1, 1))
    return mt.mean(dim=(1, 2, 3)).numpy() > cover


## 3. Get the images to test on

Pick where the demo 'training set' comes from with **`IMAGE_SOURCE`**:

- `"folder"` — **your own real images** from a Kaggle dataset or uploaded files. Add them via
  **+ Add Input** in the Kaggle sidebar, then set `IMAGE_DIR` to the folder (e.g.
  `/kaggle/input/<your-dataset>`). This is the option to use for real results on real photos.
- `"generate"` — generate a set with **Stable Diffusion** (fp16, batched prompts), the way the
  walkthrough notebook does.
- `"web"` — download a handful of sample photos (no dataset or SD weights needed).

Whichever you choose, the images are resized to 512x512 and the **same PGD attack + tile
detection** run on them in the next cells. You want at least ~8 images (4 to calibrate the
detector + 4+ to test).

> **Note on "real" poison:** the adversarial noise is injected by the notebook itself (next
> cell), because to *measure* detection you need to know which images are tampered. Using your
> own base images just makes the demo more convincing; the attack + scoring are identical.


In [ ]:
# ============================================================
# CELL 4 - Get the demo 'training set': your folder / SD-generated / web
# ============================================================
import glob

IMAGE_SOURCE = "folder"      # "folder" (your images) | "generate" (SD) | "web" (samples)
IMAGE_DIR    = "/kaggle/input"   # used when IMAGE_SOURCE=="folder"; point at your dataset
N_IMAGES     = 12            # how many images to use (need >= 8: 4 calibrate + 4+ test)
GEN_STEPS, GEN_BATCH = 25, 4 # SD generation settings (only used when IMAGE_SOURCE=="generate")

PROMPTS = [
    "a photograph of an astronaut riding a horse",
    "a watercolor painting of a fox in a snowy forest",
    "a bowl of ramen on a wooden table, studio lighting",
    "a lighthouse on a cliff at sunset, oil painting",
    "a golden retriever puppy playing in autumn leaves",
    "a red vintage car parked on a cobblestone street",
    "a hot air balloon over a mountain lake at dawn",
    "a barista pouring latte art in a cozy cafe",
    "a medieval castle on a hill under a stormy sky",
    "a close-up photo of a monarch butterfly on a flower",
    "an old sailing ship in a stormy sea, romantic painting",
    "a snowy owl perched on a fence post in winter",
]
pipe = None            # set below if we load Stable Diffusion

def _load_folder(folder, n):
    exts = (".jpg", ".jpeg", ".png", ".bmp", ".JPEG", ".JPG", ".PNG")
    paths = sorted(f for f in glob.glob(os.path.join(folder, "**", "*"), recursive=True)
                   if f.endswith(exts))
    return paths[:n]

def _load_web(n):
    base = "https://raw.githubusercontent.com/EliSchwartz/imagenet-sample-images/master/"
    names = ["n02099601_golden_retriever.JPEG", "n02123045_tabby.JPEG", "n02391049_zebra.JPEG",
             "n02129165_lion.JPEG", "n02129604_tiger.JPEG", "n02510455_giant_panda.JPEG",
             "n01518878_ostrich.JPEG", "n01806143_peacock.JPEG", "n01882714_koala.JPEG",
             "n02007558_flamingo.JPEG", "n01443537_goldfish.JPEG", "n02356798_fox_squirrel.JPEG"]
    names = names[:n]
    return [base + nm for nm in names], names

labels = None
if IMAGE_SOURCE == "folder":
    paths = _load_folder(IMAGE_DIR, N_IMAGES)
    if not paths:
        raise FileNotFoundError(
            f"No images under {IMAGE_DIR!r}. Add a dataset via '+ Add Input' and set IMAGE_DIR, "
            f"or switch IMAGE_SOURCE to 'generate' or 'web'.")
    raw = [load_image(p) for p in paths]
    labels = [os.path.basename(p) for p in paths]
    print(f"Loaded {len(raw)} real images from {IMAGE_DIR}")

elif IMAGE_SOURCE == "generate":
    from diffusers import StableDiffusionPipeline
    pipe = StableDiffusionPipeline.from_pretrained(
        "CompVis/stable-diffusion-v1-4", torch_dtype=torch.float16).to(_D)
    pipe.set_progress_bar_config(disable=True)
    g = torch.Generator(_D).manual_seed(1024)
    prompts = PROMPTS[:N_IMAGES]; pil = []
    for i in range(0, len(prompts), GEN_BATCH):
        pil += pipe(prompts[i:i+GEN_BATCH], num_inference_steps=GEN_STEPS, generator=g).images
    raw = [pil_to_tensor(im) for im in pil]
    labels = prompts
    print(f"Generated {len(raw)} images with Stable Diffusion (fp16, {GEN_STEPS} steps).")

else:  # "web"
    urls, labels = _load_web(N_IMAGES)
    raw = [load_image(u) for u in urls]
    print(f"Downloaded {len(raw)} web sample images.")

assert len(raw) >= 8, f"Need >= 8 images (got {len(raw)}): 4 to calibrate + 4+ to test."

# preview the set
ncol = 6; nrow = int(np.ceil(len(raw) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(2.4*ncol, 2.4*nrow))
for ax, x, lab in zip(np.ravel(axes), raw, labels):
    ax.imshow(to_np(x)); ax.set_title(str(lab)[:26], fontsize=7); ax.axis("off")
for ax in np.ravel(axes)[len(raw):]: ax.axis("off")
fig.suptitle(f"The demo training set  (source={IMAGE_SOURCE})", y=1.02)
plt.tight_layout(); plt.show()


## 4. Poison part of the training set

Simulates a tampered scrape: the first 4 images are held out as the **trusted clean
calibration set** (never attacked); of the remaining 'incoming' images, every other one is
poisoned with **PGD** inside a random blob.


In [ ]:
# ============================================================
# CELL 5 - Split: trusted-clean calibration set vs incoming (partly poisoned) stream
# ============================================================
ATTACK_METHOD = "pgd"    # "pgd" (stronger, subtler) or "fgsm" (1 step, fast)
EPSILON       = 0.05     # L-inf budget in real pixel units
K_SENSITIVITY = 3.0      # detector threshold: lower = more sensitive
REJECT_FRAC   = 0.10     # reject an image if > this fraction of tiles are flagged

CLEAN_CALIB = raw[:4]
THRESH = clean_threshold(CLEAN_CALIB, k=K_SENSITIVITY)
print(f"Gate calibrated on {len(CLEAN_CALIB)} trusted clean images -> threshold = {THRESH:.5f}")

incoming, truth_masks, is_poisoned, true_tiles = [], [], [], []
for i, x in enumerate(raw[4:]):
    if i % 2 == 0:
        mask = random_blob_mask(seed=i)
        incoming.append(poison(x, method=ATTACK_METHOD, epsilon=EPSILON, mask=mask))
        truth_masks.append(mask); is_poisoned.append(True)
        true_tiles.append(true_tiles_from_mask(mask))
    else:
        incoming.append(x); truth_masks.append(None); is_poisoned.append(False)
        true_tiles.append(np.zeros(GRID*GRID, bool))
print(f"Incoming stream: {len(incoming)} images, {sum(is_poisoned)} secretly poisoned "
      f"({ATTACK_METHOD.upper()}, eps={EPSILON})")


## 5. Run the gate (both T4s, one batch, no gradients)

All tiles of all incoming images go through `batched_scan` as **one stack split across the
two GPUs**. An image is **quarantined** when more than `REJECT_FRAC` of its tiles score
above the fixed clean threshold.


In [ ]:
# ============================================================
# CELL 6 - Scan + verdicts
# ============================================================
import time
t0 = time.time()
scores = batched_scan(incoming)                 # [N,16] across both GPUs
flags  = scores > THRESH                        # [N,16] per-tile decisions
frac   = flags.mean(axis=1)                     # fraction flagged per image
keep   = frac <= REJECT_FRAC                    # per-image verdict
print(f"Scanned {len(incoming)} images x {GRID*GRID} tiles on {len(DEVICES)} device(s) "
      f"in {time.time()-t0:.2f}s (forward-only, no gradients)")
print(f"Kept {int(keep.sum())} / quarantined {int((~keep).sum())}")


## 6. What the gate saw — per-image figures

Left: the incoming image (yellow outline = the *real* poison region, unknown to the gate).
Middle: per-tile noise score. Right: the gate's flagged tiles (cyan) vs truly attacked
tiles (lime). Title gives truth vs verdict.


In [ ]:
# ============================================================
# CELL 7 - Figures
# ============================================================
def _outline(ax, ids, color, ls="-"):
    for kk in np.where(ids)[0]:
        r, c = divmod(int(kk), GRID)
        ax.add_patch(patches.Rectangle((c*TILE, r*TILE), TILE, TILE,
                     fill=False, edgecolor=color, linewidth=2.5, linestyle=ls))

for i, x in enumerate(incoming):
    fig, ax = plt.subplots(1, 3, figsize=(13, 4.4))
    ax[0].imshow(to_np(x))
    if truth_masks[i] is not None:
        ax[0].contour(truth_masks[i].squeeze().numpy(), levels=[0.5],
                      colors="yellow", linewidths=2)
    ax[0].set_title("Incoming image\n(yellow = real poison region)")
    ax[1].imshow(tile_grid_to_full(norm01(scores[i])), cmap="hot")
    ax[1].set_title("Per-tile noise score")
    ax[2].imshow(to_np(x))
    _outline(ax[2], true_tiles[i], "lime"); _outline(ax[2], flags[i], "cyan", ls="--")
    ax[2].set_title("lime = TRUE poisoned tiles\ncyan-- = gate-flagged tiles")
    for a in ax: a.axis("off")
    truth = "POISONED" if is_poisoned[i] else "clean"
    verd  = "kept -> training" if keep[i] else "QUARANTINED"
    fig.suptitle(f"truth: {truth}   |   gate: {verd}   ({frac[i]:.0%} tiles flagged)",
                 y=1.02, fontsize=12)
    plt.tight_layout(); plt.show()


## 7. Scorecard

Image-level: did the gate keep the clean ones and quarantine the poisoned ones?
Tile-level (poisoned images only): precision / recall / IoU of flagged vs truly attacked
tiles — the same localization grading as the practiced notebook.


In [ ]:
# ============================================================
# CELL 8 - Scorecard table
# ============================================================
import pandas as pd
rows = []
for i in range(len(incoming)):
    t, p = true_tiles[i], flags[i]
    tp = int((t & p).sum()); fp = int((~t & p).sum()); fn = int((t & ~p).sum())
    rows.append(dict(
        image=str(labels[4+i])[:32],
        truth="poisoned" if is_poisoned[i] else "clean",
        verdict="KEEP" if keep[i] else "QUARANTINE",
        image_correct=bool(keep[i] != is_poisoned[i]),
        tiles_flagged=int(p.sum()),
        precision=round(tp/(tp+fp), 2) if tp+fp else np.nan,
        recall=round(tp/(tp+fn), 2) if tp+fn else np.nan,
        tile_IoU=round(tp/max(tp+fp+fn, 1), 2) if is_poisoned[i] else np.nan))
df = pd.DataFrame(rows)
caught = sum((not keep[i]) and is_poisoned[i] for i in range(len(incoming)))
false_q = sum((not keep[i]) and (not is_poisoned[i]) for i in range(len(incoming)))
print(f"Image accuracy: {df['image_correct'].mean():.0%}   |   "
      f"poisoned caught: {caught}/{sum(is_poisoned)}   |   "
      f"clean wrongly quarantined: {false_q}/{len(incoming)-sum(is_poisoned)}")
print(f"Mean tile-IoU on poisoned images: {df['tile_IoU'].mean():.2f}")
df


## 8. Into the Stable Diffusion training path

This is the training-side flow from the walkthrough's custom-pipeline section:
`VAE.encode -> scheduler.add_noise`. Only **kept** images proceed; the quarantined ones
never reach the VAE. The resulting noisy latents are exactly what a UNet would train on
— now guaranteed free of the poison the gate can see.


In [ ]:
# ============================================================
# CELL 9 - gate -> VAE.encode -> add diffusion noise (kept images only)
# ============================================================
try:
    from diffusers import DDPMScheduler, AutoencoderKL
    if pipe is not None:
        vae = pipe.vae                               # reuse the already-loaded fp16 VAE
    else:                                            # folder/web mode: load the VAE on its own
        vae = AutoencoderKL.from_pretrained("CompVis/stable-diffusion-v1-4",
                                            subfolder="vae", torch_dtype=torch.float16).to(_D).eval()
    sched = DDPMScheduler.from_pretrained("CompVis/stable-diffusion-v1-4",
                                          subfolder="scheduler")
    kept_imgs = [x for x, k in zip(incoming, keep) if k]
    with torch.no_grad():
        batch = torch.cat(kept_imgs).to(_D, torch.float16)
        latents = vae.encode(batch * 2 - 1).latent_dist.sample() * 0.18215   # [B,4,64,64]
        noise = torch.randn_like(latents)
        t = torch.randint(0, sched.config.num_train_timesteps,
                          (latents.shape[0],), device=_D)
        noisy_latents = sched.add_noise(latents, noise, t)
    print(f"{len(kept_imgs)} gated-clean images -> latents {tuple(latents.shape)} -> "
          f"noisy latents {tuple(noisy_latents.shape)} for UNet training.")
    print(f"{int((~keep).sum())} quarantined images never touched the VAE.")
except Exception as e:
    print("VAE step skipped (pipeline unavailable):", repr(e))
    print("The gate verdicts above are independent of this cell.")


## Recap, knobs & connection to the paper

**What this notebook adds:** the tiled FGSM/PGD noise detection I practiced in
`04-Adversarial-Attacks/` now sits as a **module inside the Stable Diffusion workflow** — every
image is tiled and scanned *before* `VAE.encode`, so the diffusion model never trains on
data the detector can tell has been tampered with.

**Why the paper matters here (ViT-ReciproCAM, arXiv:2310.02588):** the paper produces
saliency by masking spatial positions and reading the network's response — **no gradients,
no attention access, everything in one batched forward pass**. The gate borrows exactly
that property: its per-tile score is forward-only, so scanning scales to a whole training
stream (here: N images x 16 tiles in one stack, split over 2 T4s, in well under a second).

**Knobs**
- `GRID` (Cell 2) — finer grid = tighter localization, more tiles.
- `EPSILON`, `ATTACK_METHOD` (Cell 5) — poison strength; lower eps stress-tests the gate.
- `K_SENSITIVITY` — threshold `median + k*MAD`; lower = more sensitive, more false alarms.
- `REJECT_FRAC` — how many flagged tiles quarantine a whole image.

**Honest limitation (and the research direction):** the score keys on *high-frequency*
energy, which FGSM/PGD inject. A low-frequency or smoothed poison could slip through.
Next steps for a publishable study:
1. **Detection-vs-epsilon sweep** — at what perturbation strength does recall collapse?
2. Replace HF-energy with a **small learned per-tile classifier** (or the paper's actual
   ReciproCAM response) and compare on the same sweep.
3. **Close the loop:** fine-tune (e.g. LoRA) one model on gated data and one on un-gated
   poisoned data, and show the downstream difference the gate makes.
